# Plant Health Model — Train & Validate (Colab)

Trains and validates a plant-health classifier from a CSV of per-plant
vegetation indices (e.g. `plant_health_indices.csv` from the pipeline).

**Features (NDRE-priority):** NDVI, **NDRE**, TDVI, canopy height.
GNDVI is intentionally dropped so **NDRE owns the nitrogen signal** and becomes
the model's leading spectral driver (set `INCLUDE_GNDVI=True` in cell 3 to
restore the 5-feature, slightly-higher-accuracy model).
**Target:** `health_status` if present, otherwise derived from NDVI/NDRE thresholds.

It compares **RandomForest vs XGBoost**, validates on a hold-out split, and
reports accuracy, F1, a confusion matrix and feature importance.

> Run the cells top-to-bottom. Cell 2 lets you upload the CSV.

In [1]:
# 1. Install deps (Colab already has most)
!pip -q install xgboost scikit-learn pandas matplotlib seaborn


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
# 2. Load the CSV
#    In Colab this opens an upload dialog. Locally, set CSV_PATH instead.
import pandas as pd, numpy as np

CSV_PATH = None  # e.g. 'plant_health_indices.csv'
if CSV_PATH is None:
    try:
        from google.colab import files
        up = files.upload()
        CSV_PATH = list(up.keys())[0]
    except Exception:
        CSV_PATH = 'plant_health_indices.csv'

df = pd.read_csv(CSV_PATH)
print('rows:', len(df))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'plant_health_indices.csv'

In [ ]:
# 3. Choose features + target
#
# PRIORITY: NDRE (the nitrogen axis) is the feature we want the model to lean
# on most. NDRE has a correlated twin — GNDVI (corr ~0.49) — that otherwise
# "steals" the nitrogen signal and caps NDRE's importance. DROPPING GNDVI is
# the lever that actually works: it forces the model to read nitrogen from
# NDRE directly, lifting NDRE's measured importance 0.18 -> 0.21 on the
# pipeline CSV and making it the clear #2 driver (behind biomass NDVI).
#
# NOTE: we tried XGBoost `feature_weights` / column-scaling to up-weight NDRE
# and it BACKFIRED — starving trees of the other columns lowered NDRE's split
# gain and inflated canopy height instead. So the emphasis is done purely by
# curating the feature set, not by weighting. Set INCLUDE_GNDVI=True to restore
# the higher-accuracy 5-feature model (NDRE then falls back to rank 3).
INCLUDE_GNDVI = False
base_features = ['ndvi', 'ndre', 'tdvi', 'plant_height_m']
if INCLUDE_GNDVI:
    base_features.insert(2, 'gndvi')
FEATURES = [c for c in base_features if c in df.columns]
print('features (NDRE-priority set):', FEATURES)

# Target: use pipeline label if present, else derive 3 classes from NDVI/NDRE.
def derive_status(r):
    def lvl(v, hi, mid):
        if pd.isna(v): return None
        return 2 if v >= hi else (1 if v >= mid else 0)
    b, n = lvl(r.get('ndvi'), 0.6, 0.3), lvl(r.get('ndre'), 0.6, 0.2)
    if b is None or n is None: return None
    return {2: 'Healthy', 1: 'Mid Stressed', 0: 'Unhealthy'}[min(b, n)]

if 'health_status' in df.columns:
    TARGET = 'health_status'
else:
    df['health_status'] = df.apply(derive_status, axis=1)
    TARGET = 'health_status'
print('target:', TARGET)

In [ ]:
# 4. Clean: keep rows with all features + a valid, non-NOISE label
data = df.copy()
if 'sector_label' in data.columns:
    data = data[data['sector_label'] != 'NOISE']
keep_labels = ['Healthy', 'Moderate', 'Mid Stressed', 'Stressed', 'Unhealthy', 'Critical']
data = data[data[TARGET].isin(keep_labels)]
data = data.dropna(subset=FEATURES + [TARGET])
print('usable rows:', len(data))
print(data[TARGET].value_counts())

In [ ]:
# 5. Train / validation split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = data[FEATURES].values
le = LabelEncoder()
y = le.fit_transform(data[TARGET].values)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('train:', len(X_tr), ' validation:', len(X_val))

In [ ]:
# 6. Train + compare RandomForest vs XGBoost (5-fold CV on train)
#
# NDRE priority is baked into the FEATURE SET (cell 3: GNDVI removed), so the
# models here are plain — no column weighting, which we confirmed backfires.
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=8,
                                           random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, eval_metric='mlogloss'),
}
scores = {}
for name, m in models.items():
    f1 = cross_val_score(m, X_tr, y_tr, cv=cv, scoring='f1_macro')
    scores[name] = f1.mean()
    print(f'{name:<13} CV f1_macro = {f1.mean():.3f} ± {f1.std():.3f}')
best_name = max(scores, key=scores.get)
print('>>> best:', best_name)
best = models[best_name].fit(X_tr, y_tr)

# Confirm the emphasis landed: NDRE's importance + rank among the features.
_imp = dict(zip(FEATURES, best.feature_importances_))
_rank = 1 + sorted(_imp.values(), reverse=True).index(_imp['ndre'])
print(f"NDRE importance = {_imp['ndre']:.3f}  (rank {_rank}/{len(FEATURES)})")

In [ ]:
# 7. Validate on the hold-out set
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns

pred = best.predict(X_val)
print('Validation accuracy:', round(accuracy_score(y_val, pred), 3))
print(classification_report(y_val, pred, target_names=le.classes_))

cm = confusion_matrix(y_val, pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'{best_name} — confusion matrix')
plt.tight_layout(); plt.show()

In [ ]:
# 8. Feature importance
imp = pd.Series(best.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(color='seagreen'); plt.title(f'{best_name} feature importance')
plt.tight_layout(); plt.show()
print(imp[::-1])

In [ ]:
# 9. Save the trained model (download in Colab)
import pickle
with open('plant_health_model.pkl', 'wb') as f:
    pickle.dump({'model': best, 'label_encoder': le, 'features': FEATURES}, f)
print('saved -> plant_health_model.pkl')
try:
    from google.colab import files; files.download('plant_health_model.pkl')
except Exception:
    pass

## Optional — unsupervised NDVI×NDRE health zones
No labels needed: cluster plants by NDVI & NDRE and score separation with the
silhouette coefficient.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

Z = data[['ndvi', 'ndre']].dropna().values
Zs = StandardScaler().fit_transform(Z)
for k in range(2, 6):
    lab = KMeans(k, n_init=10, random_state=42).fit_predict(Zs)
    print(f'k={k}  silhouette={silhouette_score(Zs, lab, sample_size=5000, random_state=42):.3f}')

lab = KMeans(2, n_init=10, random_state=42).fit_predict(Zs)
plt.scatter(Z[:, 0], Z[:, 1], c=lab, cmap='RdYlGn', s=6, alpha=.4)
plt.xlabel('NDVI (biomass)'); plt.ylabel('NDRE (nitrogen)')
plt.title('NDVI × NDRE health zones'); plt.show()